In [1]:
# In previous lesson, lexical search has limitations  with exact text matching, so we'll need to use semantic search via
# vector embeddings to make sure we can find relevant documents even if the wording of the query has different wordings from the documents.
# 
# 1. Embed the documents into vector space using a pre-trained model.
# 2. Embed the query into the same vector space.
# 3. Compare the query embedding with the document embeddings using a similarity metric (e.g., cosine similarity) to find the most relevant documents.
# vectors pointing in the same direction will have similiarity close to 1, while vectors pointing in opposite directions will have similarity close to -1, and orthogonal vectors will have similarity close to 0.

from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [3]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)


In [4]:
v1.dot(dv)

np.float32(0.32332397)

In [5]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)
v2.dot(dv)

np.float32(0.019730438)

In [6]:
from ingest import load_faq_data
documents = load_faq_data()

In [7]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [8]:
from tqdm.auto import tqdm

In [9]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [10]:
import numpy as np
X = np.array(vectors)

In [11]:
query = "Can I still join the program after the it begins?"
v_query = model.encode(query)

In [12]:
scores = X.dot(v_query)

In [13]:
scores = [v_query.dot(X[i]) for i in range(len(X))]

In [14]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.61950815))

In [15]:
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [16]:
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]
top5

array([  2, 625, 907, 538, 495])

In [17]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.61950815
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.6057782
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.5920181
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Relate

In [18]:
from minsearch import VectorSearch
vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [19]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [20]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [21]:

results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)
results[0]


{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [22]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [23]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [37]:
from v_rag_helper import RAGVector

vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [40]:
query = "I just found out about the program, can I still sign up?"
vector_assistant.rag(query)

KeyError: 'content'